# Week 65, Day 5: State-space models of populations and behaviour

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w065/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import block_diag
g0 = np.random.default_rng(655)
T, d, N = 250, 2, 30
th = 0.07
A = 0.99 * np.array([[np.cos(th), -np.sin(th)], [np.sin(th), np.cos(th)]])
Q = 0.02 * np.eye(d)
C = 0.8 * g0.normal(size=(N, d)); dbase = np.log(0.3) * np.ones(N)
z = np.zeros((T, d)); z[0] = [1.5, 0]
for t in range(1, T):
    z[t] = A @ z[t - 1] + g0.multivariate_normal(np.zeros(d), Q)
rates = np.exp(z @ C.T + dbase)
X = g0.poisson(rates)
print(f"{X.sum()} spikes, mean {X.mean():.2f} per neuron per bin, {np.mean(X == 0):.0%} empty bins")

### Exercise: Laplace smoothing for a PLDS

Complete the gradient and Hessian of the log posterior over the whole latent path for the PLDS, and run Newton's method. The prior precision Lam is supplied; the likelihood contributes C^T (x - lambda) to the gradient and block-diagonal C^T diag(lambda) C to the Hessian.

In [ ]:
# block-tridiagonal prior precision of the latent path under the LDS prior
Qi = np.linalg.inv(Q); Ai = A.T @ Qi
Lam = np.zeros((T * d, T * d))
for t in range(T):
    blk = Qi + (A.T @ Qi @ A if t < T - 1 else np.zeros((d, d)))
    if t == 0: blk = np.linalg.inv(np.eye(d)) + A.T @ Qi @ A
    Lam[t * d:(t + 1) * d, t * d:(t + 1) * d] = blk
    if t:
        Lam[t * d:(t + 1) * d, (t - 1) * d:t * d] = -Qi @ A
        Lam[(t - 1) * d:t * d, t * d:(t + 1) * d] = -(Qi @ A).T

zc = np.zeros(T * d)
obj_hist = []
for it in range(30):
    Z = zc.reshape(T, d)
    lam = np.exp(Z @ C.T + dbase)
    ll = np.sum(X * (Z @ C.T + dbase) - lam)
    obj_hist.append(ll - 0.5 * zc @ Lam @ zc)
    grad = ...                                          # (T d,) : likelihood term minus Lam z
    H = Lam + block_diag(*[C.T @ np.diag(lam[t]) @ C for t in range(T)])
    zc = zc + np.linalg.solve(H, grad)
Zhat = zc.reshape(T, d)
U_true = np.linalg.qr(z)[0]; U_hat = np.linalg.qr(Zhat)[0]
sub_corr = np.mean(np.abs(np.linalg.svd(U_true.T @ U_hat, compute_uv=False)))
print(f"{len(obj_hist)} Newton steps; log posterior {obj_hist[0]:.1f} -> {obj_hist[-1]:.1f}")
print(f"subspace agreement with the true latent path: {sub_corr:.3f}")

In [ ]:
import numpy as np
def test_converge():
    "Newton's method converges: the log posterior rises and then stops changing"
    assert obj_hist[-1] > obj_hist[0] and abs(obj_hist[-1] - obj_hist[-2]) < 1e-6
def test_recover():
    "the inferred latent path matches the truth (subspace correlation above 0.9)"
    assert sub_corr > 0.9

test_converge()
test_recover()
print("All checks passed")

### Exercise: Poisson against Gaussian at low rates

For baseline rates of 0.3 and 5 spikes per bin, simulate counts from the same latent path, infer with the Laplace PLDS and with a Kalman smoother on square-root counts (true C and noise from the data), and compare subspace agreement with the truth.

In [ ]:
def laplace_latents(Xc, base):
    zc_ = np.zeros(T * d)
    for it in range(30):
        Z_ = zc_.reshape(T, d)
        lam_ = np.exp(Z_ @ C.T + base)
        g_ = ((Xc - lam_) @ C).reshape(-1) - Lam @ zc_
        H_ = Lam + block_diag(*[C.T @ np.diag(lam_[t]) @ C for t in range(T)])
        zc_ = zc_ + np.linalg.solve(H_, g_)
    return zc_.reshape(T, d)

def kalman_latents(Y, Cg, Rg):
    m, P = np.zeros(d), np.eye(d)
    ms, Ps, mp, Pp = [], [], [], []
    for t in range(T):
        if t: m, P = A @ m, A @ P @ A.T + Q
        mp.append(m.copy()); Pp.append(P.copy())
        S = Cg @ P @ Cg.T + Rg
        K = P @ Cg.T @ np.linalg.inv(S)
        m = m + K @ (Y[t] - Cg @ m); P = (np.eye(d) - K @ Cg) @ P
        ms.append(m.copy()); Ps.append(P.copy())
    ms, Ps, mp, Pp = map(np.array, (ms, Ps, mp, Pp))
    mh = ms.copy()
    for t in range(T - 2, -1, -1):
        J = Ps[t] @ A.T @ np.linalg.inv(Pp[t + 1])
        mh[t] = ms[t] + J @ (mh[t + 1] - mp[t + 1])
    return mh

agree = lambda Zh: np.mean(np.abs(np.linalg.svd(np.linalg.qr(z)[0].T @ np.linalg.qr(Zh)[0], compute_uv=False)))
gc = np.random.default_rng(6551)
res = {}
for base_rate in (0.3, 5.0):
    base = np.log(base_rate) * np.ones(N)
    Xr = gc.poisson(np.exp(z @ C.T + base))
    Zp = laplace_latents(Xr, base)
    Y = np.sqrt(Xr); Y = Y - Y.mean(0)
    Cg = np.linalg.lstsq(z, Y, rcond=None)[0].T                      # loadings for the square-root counts
    Rg = np.diag(np.var(Y - z @ Cg.T, axis=0))
    Zg = kalman_latents(Y, Cg, Rg)
    res[base_rate] = {"poisson": agree(Zp), "gauss": ...}
    print(f"baseline {base_rate} spikes per bin: Poisson {res[base_rate]['poisson']:.3f}, Gaussian on sqrt counts {res[base_rate]['gauss']:.3f}")

In [ ]:
import numpy as np
def test_low():
    "at low firing rates the Poisson model recovers the latent better"
    assert res[0.3]["poisson"] > res[0.3]["gauss"] + 0.02
def test_high():
    "at high rates the two are close"
    assert abs(res[5.0]["poisson"] - res[5.0]["gauss"]) < 0.03

test_low()
test_high()
print("All checks passed")

### Exercise: Segmenting behaviour into regimes

Simulate 1,500 steps of a one-dimensional behavioural variable whose dynamics switch between three regimes. Complete the emission likelihoods (the one-step prediction density of each regime), run forward-backward to infer the regimes, and update each regime's parameters by weighted least squares. Repeat for 25 EM iterations.

In [ ]:
gs = np.random.default_rng(6552)
Tb = 1500
truth = [(0.5, 0.0, 0.05), (0.95, 0.3, 0.05), (0.7, -0.4, 0.25)]      # (decay, drift, noise variance)
P_sw = np.full((3, 3), 0.01); np.fill_diagonal(P_sw, 0.98)
s_true = np.zeros(Tb, int); v = np.zeros(Tb)
for t in range(1, Tb):
    s_true[t] = gs.choice(3, p=P_sw[s_true[t - 1]])
    a_, b_, q_ = truth[s_true[t]]
    v[t] = a_ * v[t - 1] + b_ + np.sqrt(q_) * gs.normal()

a_hat, b_hat, q_hat = [0.8, 0.8, 0.8], [0.0, 0.1, -0.1], [0.2, 0.2, 0.2]
Phat = np.full((3, 3), 0.05); np.fill_diagonal(Phat, 0.9)
for it in range(25):
    B = np.zeros((Tb - 1, 3))
    for k in range(3):
        pred = a_hat[k] * v[:-1] + b_hat[k]
        B[:, k] = ...                                   # density of v[1:] under regime k
    B += 1e-300
    al = np.zeros((Tb - 1, 3)); c = np.zeros(Tb - 1); be = np.ones((Tb - 1, 3))
    a0 = np.full(3, 1 / 3) * B[0]; c[0] = a0.sum(); al[0] = a0 / c[0]
    for t in range(1, Tb - 1):
        a0 = B[t] * (al[t - 1] @ Phat); c[t] = a0.sum(); al[t] = a0 / c[t]
    for t in range(Tb - 3, -1, -1):
        be[t] = (Phat @ (B[t + 1] * be[t + 1])) / c[t + 1]
    gam = al * be; gam /= gam.sum(1, keepdims=True)
    xi = al[:-1, :, None] * Phat[None] * (B[1:] * be[1:])[:, None, :]
    xi /= xi.sum((1, 2), keepdims=True)
    Phat = xi.sum(0) / xi.sum(0).sum(1, keepdims=True)
    for k in range(3):
        w = gam[:, k]
        Xd = np.column_stack([v[:-1], np.ones(Tb - 1)])
        sol = np.linalg.solve(Xd.T @ (Xd * w[:, None]), Xd.T @ (w * v[1:]))
        a_hat[k], b_hat[k] = sol
        q_hat[k] = max(np.sum(w * (v[1:] - Xd @ sol) ** 2) / w.sum(), 1e-4)
seg = gam.argmax(1)
order = np.argsort(a_hat)
acc_seg = max(np.mean(np.array(perm)[seg] == s_true[1:]) for perm in __import__("itertools").permutations(range(3)))
print(f"segmentation accuracy {acc_seg:.3f}")
print("learned (decay, drift, noise):", [(round(float(a_hat[k]), 2), round(float(b_hat[k]), 2), round(float(q_hat[k]), 3)) for k in order])
print("true:                        ", [tuple(round(x, 2) for x in truth[k]) for k in np.argsort([t[0] for t in truth])])

In [ ]:
import numpy as np
def test_segment():
    "the inferred regimes match the true ones on more than 85% of time steps"
    assert acc_seg > 0.85
def test_params():
    "the learned regime dynamics separate the three motifs"
    assert len(set(np.round(np.array(a_hat), 1))) == 3

test_segment()
test_params()
print("All checks passed")